In [1]:
# Mount Google Drive to access dataset files stored in Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Import core libraries for numerical operations (numpy) and data manipulation (pandas)
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

In [3]:
# Load cleanver csv from Google Drive into a pandas DataFrame
# Preview first few rows to verify successful loading and structure

csv_file = '/content/drive/My Drive/EADA/EADA_DeepLearning/Project/cleanver_books_dataset.csv'

df = pd.read_csv(csv_file)

df.head()

,book_id,title,subtitle,authors,publisher,published_date,description,page_count,categories,average_rating,...,list_price,currency,buyable,search_category,thumbnail,title_clean,reading_time,length_category,combined_text,category_source
0,LR_VDQAAQBAJ,bestsellers,"the path (bestsellers, free bestsellers, bests...","['ivan king', '']",bestsellers,2017-01-04,"Hear What the Critics are Saying ""Wow, what an...",70,['young adult fiction'],NaN,...,12.99,USD,True,fiction bestsellers,http://books.google.com/books/content?id=LR_VD...,bestsellers,2.333333,short,"bestsellers the path (bestsellers, free bestse...",original
1,WcjTDQAAQBAJ,bestsellers,"hell: a place without hope (bestseller books, ...","['ivan king', '']",bestsellers,2017-01-03,"Hear What the Critics are Saying Wow, very ins...",32,"['comics', 'graphic novels']",NaN,...,9.99,USD,True,fiction bestsellers,http://books.google.com/books/content?id=WcjTD...,bestsellers,1.066667,short,bestsellers hell: a place without hope (bestse...,original
2,4fXUDAAAQBAJ,the bestseller code,anatomy of the blockbuster novel,"['jodie archer', 'matthew l jockers']",macmillan,2016-09-20,"""What if there was an algorithm that could pre...",253,"['business', 'economics']",NaN,...,NaN,NaN,False,fiction bestsellers,http://books.google.com/books/content?id=4fXUD...,thebestsellercode,8.433333,medium,the bestseller code anatomy of the blockbuster...,original
3,yIVuDwAAQBAJ,bestseller,a century of america's favorite books,['robert mcparland'],bloomsbury publishing plc,2018-12-15,Whether curled up on a sofa with a good myster...,335,['literary criticism'],NaN,...,40.50,USD,True,fiction bestsellers,http://books.google.com/books/content?id=yIVuD...,bestseller,11.166667,medium,bestseller a century of america's favorite boo...,original
4,2JHXwAEACAAJ,bestsellers: popular fiction since 1900,NaN,['c bloom'],palgrave macmillan,2002-07-09,This guide and reference work of all of the be...,306,['literary criticism'],NaN,...,NaN,NaN,False,fiction bestsellers,http://books.google.com/books/content?id=2JHXw...,bestsellerspopularfictionsince1900,10.200000,medium,bestsellers: popular fiction since 1900 nan c....,original


In [4]:
def fetch_book_info(info_link):
# Function to scrape metadata (publish date, rating, ratings count)
# from a Google Books webpage using the provided URL (info_link)

    try:
# Send HTTP request to the book's Google Books page
# timeout=5 prevents the request from hanging too long
        r = requests.get(info_link, timeout=5)
# Parse the HTML content of the page using BeautifulSoup
        soup = BeautifulSoup(r.text, 'html.parser')

# Extract all visible text from the page and convert to lowercase
# This simplifies pattern matching later (case-insensitive search)
        text = soup.get_text().lower()
# Initialize default values (in case extraction fails)
# crude extraction (works as fallback)
        pub_date = None
        rating = None
        ratings_count = None

        # Extract patterns
        date_match = re.search(r'published.*?(\d{4})', text)
        rating_match = re.search(r'(\d\.\d)\s*out of 5', text)
        count_match = re.search(r'(\d+,?\d*)\s+ratings', text)

        if date_match:
# Extract year (as string)
            pub_date = date_match.group(1)

        if rating_match:
 # Convert rating string (e.g., "4.5") to float
            rating = float(rating_match.group(1))

        if count_match:
# Remove commas and convert to integer (e.g., "1,234" → 1234)
            ratings_count = int(count_match.group(1).replace(',', ''))
# Return extracted values (or None if not found)
        return pub_date, rating, ratings_count

    except:
# If ANY error occurs (network issue, parsing issue, etc.)
# return None values to avoid crashing the pipeline
        return None, None, None

In [5]:
# Apply ONLY to missing fields
def fill_missing_metadata(row):
# Check if any of the target fields are missing
    if pd.isna(row['published_date']) or pd.isna(row['average_rating']) or pd.isna(row['ratings_count']):
# Fetch missing data from external source (Google Books page)
        pub, rate, count = fetch_book_info(row['info_link'])

# Return updated values:
# - Use fetched value ONLY if original is missing
# - Otherwise keep existing value
        return pd.Series([
            pub if pd.isna(row['published_date']) else row['published_date'],
            rate if pd.isna(row['average_rating']) else row['average_rating'],
            count if pd.isna(row['ratings_count']) else row['ratings_count']
        ])
    else:
        return pd.Series([row['published_date'], row['average_rating'], row['ratings_count']])

# Apply the function row-by-row across the dataframe
# Update the three columns with enriched values
df[['published_date','average_rating','ratings_count']] = df.apply(fill_missing_metadata, axis=1)

In [6]:
# Create a boolean column:
# True  → book has a price (buyable)
# False → no price available
df['is_buyable'] = df['list_price'].notna()

# Fill missing prices for buyable books (fallback strategy)
df['list_price'] = df.apply(

    # lambda = inline function applied to each row
    lambda row:
        # If book is NOT buyable → force price to NaN
        np.nan if not row['is_buyable']

        # If book is buyable:
        else (
            # Keep existing price if already present
            row['list_price']

            # Otherwise fill missing price with dataset median
            if not pd.isna(row['list_price'])
            else df['list_price'].median()
        ),

    axis=1  # axis=1 → apply function row-by-row (not column-wise)
)

In [7]:
# Optional: flag inconsistencies
df['price_flag'] = df.apply(
    lambda row: 'missing_price' if row['is_buyable'] and pd.isna(row['list_price']) else 'ok',
    axis=1
)

In [8]:
# Existing = assumed 30 pages/hour
df['reading_time_fast'] = df['page_count'] / 30

# New = slow reader (20 pages/hour)
df['reading_time_slow'] = df['page_count'] / 20

In [9]:
# Function that fills missing values in the "length_category" column
def fix_length_category(row):
    # "row" represents one row of the DataFrame at a time (when using apply with axis=1)

    # Check if length_category is missing (NaN)
    if pd.isna(row['length_category']):

        # If page_count is 150 or less, classify as 'short'
        if row['page_count'] <= 150:
            return 'short'

        # If page_count is between 151 and 400, classify as 'medium'
        elif row['page_count'] <= 400:
            return 'medium'

        # If page_count is above 400, classify as 'long'
        else:
            return 'long'

    # If length_category is already filled, keep the original value
    return row['length_category']


# Apply the function to each row in the DataFrame
# axis=1 means we apply it row by row (not column by column)
df['length_category'] = df.apply(fix_length_category, axis=1)

In [10]:
mood_keywords = {
    # Expanded original categories
    'dark': ['death', 'war', 'murder', 'crime', 'loss', 'betrayal', 'tragedy', 'grief', 'sinister', 'macabre', 'horror', 'revenge'],
    'inspiring': ['success', 'growth', 'achievement', 'motivation', 'triumph', 'hope', 'perseverance', 'courage', 'overcoming', 'resilience'],
    'romantic': ['love', 'relationship', 'heart', 'romance', 'passion', 'kiss', 'lovers', 'desire', 'marriage', 'soulmate', 'infatuation'],
    'adventurous': ['journey', 'quest', 'explore', 'adventure', 'expedition', 'discovery', 'voyage', 'epic', 'survival', 'frontier', 'escape'],
    'educational': ['guide', 'learn', 'introduction', 'study', 'history', 'science', 'theory', 'analysis', 'concept', 'textbook', 'nonfiction'],
    'thrilling': ['mystery', 'suspense', 'thriller', 'danger', 'espionage', 'conspiracy', 'detective', 'secrets', 'tension', 'plot-twist', 'stakes'],

    # New categories
    'funny': ['humor', 'comedy', 'hilarious', 'laugh', 'satire', 'witty', 'joke', 'parody', 'amusing', 'lighthearted'],
    'magical': ['magic', 'fantasy', 'witches', 'dragons', 'spells', 'wizard', 'supernatural', 'enchantment', 'myth', 'fairy-tale'],
    'melancholic': ['sadness', 'nostalgia', 'lonely', 'sorrow', 'heartbreak', 'regret', 'bittersweet', 'longing', 'tearjerker', 'gloomy'],
    'philosophical': ['meaning', 'existence', 'truth', 'ethics', 'thought', 'consciousness', 'morality', 'wisdom', 'reflection', 'society'],
    'futuristic': ['future', 'space', 'technology', 'alien', 'galaxy', 'cyberpunk', 'robot', 'dystopia', 'sci-fi', 'universe'],
    'cozy': ['cozy', 'tea', 'baking', 'small-town', 'comfort', 'peaceful', 'gentle', 'wholesome', 'healing', 'friendship']
}

# Function that detects the "mood" of a text based on keyword matching
def detect_mood(text):
    # Convert input text to string and make it lowercase
    # This ensures matching is consistent (case-insensitive)
    text = str(text).lower()

    # Dictionary to store how many keyword matches each mood gets
    scores = {}

    # Loop through each mood and its associated keyword list
    # mood_keywords is assumed to be a dictionary like:
    # {"happy": ["good", "great"], "sad": ["bad", "sad"]}
    for mood, words in mood_keywords.items():

        # Count how many keywords appear in the text
        # (True counts as 1, False as 0)
        scores[mood] = sum(w in text for w in words)

    # Find the mood with the highest score (most keyword matches)
    best_mood = max(scores, key=scores.get)

    # If no keywords matched (score = 0), return 'neutral'
    # Otherwise return the detected mood
    return best_mood if scores[best_mood] > 0 else 'neutral'


# Apply the function to each row in the "combined_text" column
# axis=1 is NOT needed here because we're working column-wise (Series apply)
df['mood'] = df['combined_text'].apply(detect_mood)

In [11]:
# Save cleaned csv
df.to_csv('/content/drive/My Drive/EADA/EADA_DeepLearning/Project/cleanver2_books_dataset.csv', index=False)